In [2]:
!pip install langchain langchain-community langchain-core
!pip install langchain-text-splitters
!pip install langchain-huggingface
!pip install langchain-chroma
!pip install langchain-ollama
!pip install chromadb
!pip install pypdf


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load the PDF Document

In [30]:
import ipywidgets as widgets
from IPython.display import display
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF file path input
pdf_path = widgets.Text(
    description="PDF path:",
    placeholder="Paste the full path of your PDF here",
    layout=widgets.Layout(width="80%")
)

load_button = widgets.Button(description="Load PDF")
output = widgets.Output()

def load_pdf(b):
    with output:
        output.clear_output()

        try:
            loader = PyPDFLoader(pdf_path.value.strip())
            documents = loader.load()

            print(f"Loaded {len(documents)} pages successfully!")

        except Exception as e:
            print("Error loading PDF:", e)

load_button.on_click(load_pdf)

display(pdf_path, load_button, output)

Text(value='', description='PDF path:', layout=Layout(width='80%'), placeholder='Paste the full path of your P…

Button(description='Load PDF', style=ButtonStyle())

Output()

## 2. Splitting documents into chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks= splitter.split_documents(documents)

## 3. Create Embeddings

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

print("Embedding model loaded successfully!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\ADMIN\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


## 4. Create vector database

In [12]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents = chunks,
    embedding=embeddings,
    persist_directory="./local_chroma_db"
)
print("Vector database created")

Vector database created


## 5. Setup the retriever

In [13]:
retriever = vector_store.as_retriever(
    search_kwargs={"k":3}
)

## 6. Connect to Local LLM

In [ ]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model = "qwen3:0.6b"
)

# 7. Define the Prompt Template

In [27]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
You are a helpful AI assistant.

Use the following retrieved context to answer the question.

Context:
{context}

Question:
{question}

Answer concisely in no more than three sentences.
""" 
)

## 8. Build Rag Pipeline

In [28]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
| prompt
| llm
| StrOutputParser()
)

In [29]:
print("\nLocal RAG System Ready!")
print("Type 'exit' to quit\n")

while True:
    question = input("Ask a question: ")

    if question.lower() == "exit":
        break

    response = rag_chain.invoke(question)

    print("\nAnswer:")
    print(response)
    print("\n")


Local RAG System Ready!
Type 'exit' to quit



Ask a question:  Guidelines for Individual Participants



Answer:
Guidelines for Individual Participants require automatic approval upon registration and provide login credentials from the ISRO LMS. Participants can attend live sessions via the IIRS YouTube Channel and access features like Course Guidelines, Video Sessions, and daily quizzes. Dedication is tracked via engagement metrics, and a certificate is awarded after meeting minimum training dedication hours, accessible directly from the LMS.




Ask a question:  exit
